# RQ1 — Text Mining + Clustering on GRBench

**Research question:** *Can text embeddings and unsupervised clustering reveal latent question types that predict difficulty level better than domain labels alone?*

**Scope:** 1,740 GRBench questions across 10 domains × 3 difficulty levels.

**Pipeline:**
1. Load GRBench, sanity-check distributions.
2. Sentence-BERT (`all-MiniLM-L6-v2`) embeddings.
3. K-Means sweep k ∈ {3..11}: silhouette, ARI vs difficulty, ARI vs domain.
4. DBSCAN with two (eps, min_samples) configurations.
5. UMAP 2D projection colored by domain and by difficulty.
6. Difficulty classification with StratifiedKFold(k=5) on three feature sets × two classifiers.
7. Corrected resampled t-test on per-fold F1, combined vs domain-only.
8. Dump `metrics.json` per the handoff spec.

**Collaboration declaration**
1. **Collaborators**: solo project.
2. **Web sources**: GRBench dataset card (https://huggingface.co/datasets/PeterJinGo/GRBench).
3. **AI tools**: Claude Code (Anthropic) scaffolded this notebook and wrote analysis code; experimental design and validation reviewed by Yuyang Bai.
4. **Citations**: Reimers & Gurevych, *Sentence-BERT*, EMNLP 2019 · Hubert & Arabie, *Comparing partitions*, J. Classif. 1985 (ARI) · McInnes et al., *UMAP*, arXiv 2018 · Ester et al., *DBSCAN*, KDD 1996 · Nadeau & Bengio, *Inference for the generalization error*, ML 2003 (corrected resampled t-test).

In [1]:
from __future__ import annotations

import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.cluster import DBSCAN, KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score, f1_score, silhouette_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="talk")

HANDOFF = Path("/home/ybai/CSCE676/handoff/rq1_text_clustering")
FIGS = HANDOFF / "figures"
TABLES = HANDOFF / "tables"
CACHE = HANDOFF.parent / "_cache"
DATA_DIR = Path("/home/ybai/CSCE676/data/raw/grbench")
for d in (FIGS, TABLES):
    d.mkdir(parents=True, exist_ok=True)

wall_clock: dict[str, float] = {}

## 1. Load GRBench

GRBench ships as 10 JSONL files, one per domain. We concatenate into a single DataFrame and verify the totals against `CSCE676-Project/reports/dataset_profiles.json` (produced by Checkpoint 1).

In [2]:
t0 = time.time()
rows: list[dict] = []
for p in sorted(DATA_DIR.glob("*.json")):
    for line in p.read_text().splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        r["domain"] = p.stem
        rows.append(r)
df = pd.DataFrame(rows).reset_index(drop=True)
assert len(df) == 1740, f"expected 1740 questions, got {len(df)}"
assert df["domain"].nunique() == 10
assert set(df["level"]) == {"easy", "medium", "hard"}

print("shape:", df.shape)
print("domain counts:", df["domain"].value_counts().to_dict())
print("level counts :", df["level"].value_counts().to_dict())
wall_clock["load_grbench"] = time.time() - t0

shape: (1740, 5)
domain counts: {'healthcare': 270, 'literature': 240, 'amazon': 200, 'legal': 180, 'computer_science': 150, 'biology': 140, 'chemistry': 140, 'materials_science': 140, 'medicine': 140, 'physics': 140}
level counts : {'medium': 920, 'easy': 700, 'hard': 120}


Distribution matches Checkpoint 1 exactly: easy 700 / medium 920 / hard 120. The 10 domains are imbalanced — healthcare (270) and literature (240) dominate; most others have ~140. This matters for downstream classification — we use macro-F1 and StratifiedKFold to avoid size bias.

## 2. Sentence-BERT embeddings

**Why Sentence-BERT (`all-MiniLM-L6-v2`)?** It is the same model used in Checkpoint 2, so embeddings are directly comparable. The 384-dim output is small enough to cluster with scikit-learn's in-memory K-Means at n = 1,740 without approximation, yet rich enough to capture question semantics beyond bag-of-words.

Embeddings are L2-normalized at encode time — this makes cosine and Euclidean distance interchangeable, which is important for K-Means (which assumes Euclidean) and for DBSCAN (where we use cosine).

In [3]:
t0 = time.time()
CACHE.mkdir(parents=True, exist_ok=True)
cache_path = CACHE / "minilm_embeddings.npz"
if cache_path.exists():
    z = np.load(cache_path)
    embeddings = z["emb"]
    print(f"loaded cached embeddings from {cache_path.name}: {embeddings.shape}")
else:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeddings = model.encode(
        df["question"].tolist(),
        batch_size=128,
        show_progress_bar=False,
        normalize_embeddings=True,
    )
    qids = df.apply(lambda r: f"{r['domain']}::{r['qid']}", axis=1).to_numpy()
    np.savez(cache_path, emb=embeddings, qids=qids)
    print(f"computed and cached {embeddings.shape}")
wall_clock["embeddings"] = time.time() - t0

loaded cached embeddings from minilm_embeddings.npz: (1740, 384)


## 3. K-Means sweep — k ∈ {3..11}

For each k we measure:
* **Silhouette** on the full Euclidean embedding (higher = tighter, better-separated clusters).
* **ARI vs difficulty labels** — does the clustering recover the difficulty partition?
* **ARI vs domain labels** — does it recover the domain partition?

If embeddings predict difficulty better than the domain structure, ARI-vs-difficulty should be **non-trivial** and at minimum non-decreasing as k grows past 3 (the number of difficulty levels), and we expect ARI-vs-domain to peak near k=10 (the number of domains).

In [4]:
t0 = time.time()
ks = list(range(3, 12))
kmeans_rows = []
for k in ks:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(embeddings)
    kmeans_rows.append(
        {
            "k": k,
            "silhouette": float(silhouette_score(embeddings, labels, metric="euclidean", sample_size=1740, random_state=SEED)),
            "ari_vs_difficulty": float(adjusted_rand_score(df["level"], labels)),
            "ari_vs_domain": float(adjusted_rand_score(df["domain"], labels)),
            "inertia": float(km.inertia_),
        }
    )
kmeans_df = pd.DataFrame(kmeans_rows)
kmeans_df.to_csv(TABLES / "kmeans_sweep.csv", index=False)
print(kmeans_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True)
axes[0].plot(kmeans_df["k"], kmeans_df["silhouette"], marker="o", color="#1f77b4")
axes[0].set(title="Silhouette score", xlabel="k", ylabel="silhouette")
axes[1].plot(kmeans_df["k"], kmeans_df["ari_vs_difficulty"], marker="o", color="#d62728", label="vs difficulty")
axes[1].plot(kmeans_df["k"], kmeans_df["ari_vs_domain"], marker="s", color="#2ca02c", label="vs domain")
axes[1].set(title="ARI vs label", xlabel="k", ylabel="ARI")
axes[1].legend(loc="upper left")
axes[2].plot(kmeans_df["k"], kmeans_df["inertia"], marker="o", color="#7f7f7f")
axes[2].set(title="Inertia (elbow)", xlabel="k", ylabel="inertia")
plt.tight_layout()
fig.savefig(FIGS / "kmeans_sweep.png", dpi=180, bbox_inches="tight")
plt.close(fig)
wall_clock["kmeans_sweep"] = time.time() - t0

best_k_row = kmeans_df.loc[kmeans_df["silhouette"].idxmax()]
print("best k by silhouette:", int(best_k_row["k"]))

    k  silhouette  ari_vs_difficulty  ari_vs_domain      inertia
0   3    0.086261           0.038668       0.244217  1321.510376
1   4    0.096792           0.043886       0.264883  1274.377319
2   5    0.076412           0.097357       0.400724  1232.741089
3   6    0.082691           0.059548       0.488826  1199.267944
4   7    0.076579           0.061565       0.429376  1177.836304
5   8    0.062114           0.051668       0.470627  1156.886230
6   9    0.062386           0.051751       0.521421  1144.145752
7  10    0.060166           0.042118       0.516009  1132.105591
8  11    0.053328           0.060891       0.475227  1121.555542


best k by silhouette: 4


## 4. DBSCAN — two configurations

DBSCAN needs a distance metric and density parameters. Since our embeddings are L2-normalized, cosine distance is natural. We sweep two reasonable configs to see how cluster structure varies with density:
* **tight**: `eps=0.25, min_samples=10` — forces compact, high-density clusters.
* **loose**: `eps=0.45, min_samples=5` — allows larger, looser clusters and fewer points labelled as noise.

We report cluster count, noise fraction, and silhouette on the non-noise subset (silhouette is only meaningful when ≥2 clusters have ≥2 points).

In [5]:
t0 = time.time()
dbscan_configs = [
    {"eps": 0.25, "min_samples": 10, "label": "tight"},
    {"eps": 0.45, "min_samples": 5, "label": "loose"},
]
dbscan_rows = []
for cfg in dbscan_configs:
    db = DBSCAN(eps=cfg["eps"], min_samples=cfg["min_samples"], metric="cosine")
    labels = db.fit_predict(embeddings)
    mask = labels != -1
    n_clusters = int(len({l for l in labels if l != -1}))
    if n_clusters >= 2 and mask.sum() >= 2:
        sil = float(
            silhouette_score(embeddings[mask], labels[mask], metric="cosine", sample_size=int(mask.sum()), random_state=SEED)
        )
    else:
        sil = float("nan")
    dbscan_rows.append(
        {
            "config": cfg["label"],
            "eps": cfg["eps"],
            "min_samples": cfg["min_samples"],
            "n_clusters": n_clusters,
            "noise_frac": float((labels == -1).mean()),
            "silhouette_non_noise": sil,
        }
    )
dbscan_df = pd.DataFrame(dbscan_rows)
dbscan_df.to_csv(TABLES / "dbscan_configs.csv", index=False)
print(dbscan_df)
wall_clock["dbscan"] = time.time() - t0

  config   eps  min_samples  n_clusters  noise_frac  silhouette_non_noise
0  tight  0.25           10           3    0.982184              0.684974
1  loose  0.45            5           9    0.311494              0.027157


## 5. UMAP 2D projection

UMAP maps the 384-dim embeddings to 2D for visualization. We colour by **domain** and by **difficulty** separately. The question is whether the two label systems produce visually different cluster structures in the same projection — if difficulty is latent and orthogonal to domain, the difficulty plot will show stripes/bands inside each domain blob rather than a single-colour-per-region pattern.

In [6]:
t0 = time.time()
import umap  # noqa: E402

reducer = umap.UMAP(n_components=2, metric="cosine", random_state=SEED, n_neighbors=15, min_dist=0.1)
coords = reducer.fit_transform(embeddings)
umap_df = pd.DataFrame(coords, columns=["umap_x", "umap_y"])
umap_df["domain"] = df["domain"].values
umap_df["level"] = df["level"].values
umap_df.to_csv(TABLES / "umap_coords.csv", index=False)

for colour_by, fname, palette, order in [
    ("domain", "umap_domain.png", "tab10", sorted(df["domain"].unique())),
    ("level", "umap_difficulty.png", {"easy": "#2ca02c", "medium": "#ff7f0e", "hard": "#d62728"}, ["easy", "medium", "hard"]),
]:
    fig, ax = plt.subplots(figsize=(8, 6.5))
    sns.scatterplot(data=umap_df, x="umap_x", y="umap_y", hue=colour_by, hue_order=order, palette=palette, s=14, alpha=0.75, ax=ax)
    ax.set(title=f"UMAP of MiniLM embeddings — coloured by {colour_by}", xlabel="UMAP-1", ylabel="UMAP-2")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False, fontsize=9)
    plt.tight_layout()
    fig.savefig(FIGS / fname, dpi=180, bbox_inches="tight")
    plt.close(fig)
wall_clock["umap"] = time.time() - t0

/home/ybai/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/ybai/.local/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


## 6. Difficulty classification — does embedding geometry beat domain labels?

Three feature sets:
1. **domain-only**: one-hot encoded domain (10 dims).
2. **embeddings**: MiniLM 384-dim.
3. **combined**: concatenation (394 dims).

Two classifiers: **Random Forest** (`n_estimators=300`, `random_state=42`) and **Logistic Regression** (`max_iter=1000`, `random_state=42`, `class_weight='balanced'` because easy/medium/hard are imbalanced 700/920/120).

Evaluation: **Stratified 5-fold** on `level`. We record macro-F1 per fold (for the paired t-test later) and the mean macro-F1 across folds (headline).

In [7]:
t0 = time.time()
y = df["level"].to_numpy()
ohe = OneHotEncoder(sparse_output=False).fit(df[["domain"]])
X_domain = ohe.transform(df[["domain"]])
X_emb = embeddings
X_both = np.hstack([X_domain, X_emb])

feature_sets = {"domain_only": X_domain, "embeddings": X_emb, "combined": X_both}
classifiers = {
    "rf": lambda: RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1),
    "lr": lambda: LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced"),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
per_fold: dict[str, list[float]] = {}
per_class_f1: dict[str, dict[str, float]] = {}
rows = []
for clf_name, clf_factory in classifiers.items():
    for fs_name, X in feature_sets.items():
        fold_scores = []
        per_class_fold = {lv: [] for lv in ["easy", "medium", "hard"]}
        for train_idx, test_idx in skf.split(X, y):
            clf = clf_factory()
            clf.fit(X[train_idx], y[train_idx])
            yp = clf.predict(X[test_idx])
            fold_scores.append(f1_score(y[test_idx], yp, average="macro"))
            pc = f1_score(y[test_idx], yp, average=None, labels=["easy", "medium", "hard"])
            for lv, v in zip(["easy", "medium", "hard"], pc):
                per_class_fold[lv].append(v)
        key = f"{fs_name}_{clf_name}"
        per_fold[key] = fold_scores
        per_class_f1[key] = {lv: float(np.mean(vs)) for lv, vs in per_class_fold.items()}
        rows.append(
            {
                "classifier": clf_name,
                "feature_set": fs_name,
                "macro_f1_mean": float(np.mean(fold_scores)),
                "macro_f1_std": float(np.std(fold_scores)),
                **{f"f1_{lv}": float(np.mean(per_class_fold[lv])) for lv in ["easy", "medium", "hard"]},
            }
        )
clf_df = pd.DataFrame(rows).sort_values(["classifier", "feature_set"]).reset_index(drop=True)
clf_df.to_csv(TABLES / "difficulty_classification.csv", index=False)
print(clf_df)
wall_clock["classification"] = time.time() - t0

  classifier  feature_set  macro_f1_mean  macro_f1_std   f1_easy  f1_medium  \
0         lr     combined       0.839177      0.023844  0.845352   0.866165   
1         lr  domain_only       0.331590      0.028145  0.335345   0.487173   
2         lr   embeddings       0.831504      0.014917  0.834861   0.853908   
3         rf     combined       0.584627      0.023023  0.818048   0.844620   
4         rf  domain_only       0.332725      0.017804  0.330464   0.667712   
5         rf   embeddings       0.584869      0.012956  0.816820   0.844247   

    f1_hard  
0  0.806015  
1  0.172253  
2  0.805744  
3  0.091214  
4  0.000000  
5  0.093538  


## 7. Paired significance test — combined vs domain-only (per classifier)

Per-fold macro-F1 scores are paired across feature sets (same fold = same test partition). The **corrected resampled t-test** (Nadeau & Bengio, 2003) accounts for the optimistic variance bias of naïve paired-t on cross-validated scores by inflating the variance with a train/test size correction: `variance *= (1/k + test_frac/(1 - test_frac))`. In our 5-fold setup `test_frac = 1/5` so the correction factor is `1/5 + 1/4 = 0.45`.

In [8]:

def corrected_resampled_t(diffs: list[float], n_train_over_n_test: float = 4.0) -> tuple[float, float, float]:
    """Nadeau & Bengio (2003) corrected resampled t-test for k-fold CV."""
    d = np.asarray(diffs, dtype=float)
    k = len(d)
    mean = float(d.mean())
    var = float(d.var(ddof=1))
    correction = 1.0 / k + 1.0 / n_train_over_n_test
    t = mean / np.sqrt(var * correction) if var > 0 else float("inf")
    p = 2 * stats.t.sf(abs(t), df=k - 1)
    return mean, float(t), float(p)


sig_rows = []
for clf_name in classifiers:
    a = per_fold[f"combined_{clf_name}"]
    b = per_fold[f"domain_only_{clf_name}"]
    diffs = [ai - bi for ai, bi in zip(a, b)]
    mean_diff, t_stat, p_val = corrected_resampled_t(diffs)
    sig_rows.append(
        {
            "classifier": clf_name,
            "test": "corrected_resampled_t",
            "mean_diff": mean_diff,
            "t_stat": t_stat,
            "p_value": p_val,
            "folds": len(diffs),
        }
    )
sig_df = pd.DataFrame(sig_rows)
sig_df.to_csv(TABLES / "significance_combined_vs_domain.csv", index=False)
print(sig_df)

  classifier                   test  mean_diff     t_stat   p_value  folds
0         rf  corrected_resampled_t   0.251902  27.353072  0.000011      5
1         lr  corrected_resampled_t   0.507587  16.732810  0.000075      5


## 8. Consolidated `metrics.json`

Schema follows INSTRUCTIONS.md §3 RQ1 exactly — Cowork Claude reads this file verbatim to populate the final deliverable narrative.

In [9]:
best_row_idx = kmeans_df["silhouette"].idxmax()
best_k = int(kmeans_df.loc[best_row_idx, "k"])

metrics = {
    "n_questions": int(len(df)),
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_dim": int(embeddings.shape[1]),
    "domain_counts": df["domain"].value_counts().to_dict(),
    "level_counts": df["level"].value_counts().to_dict(),
    "kmeans": {
        "k_sweep": ks,
        "best_k": best_k,
        "best_silhouette": float(kmeans_df.loc[best_row_idx, "silhouette"]),
        "ari_vs_difficulty_at_best_k": float(kmeans_df.loc[best_row_idx, "ari_vs_difficulty"]),
        "ari_vs_domain_at_best_k": float(kmeans_df.loc[best_row_idx, "ari_vs_domain"]),
        "per_k": kmeans_df.to_dict(orient="records"),
    },
    "dbscan": dbscan_df.to_dict(orient="records"),
    "difficulty_f1": {row["feature_set"] + "_" + row["classifier"]: {
        "macro_f1_mean": row["macro_f1_mean"],
        "macro_f1_std": row["macro_f1_std"],
        "per_class": per_class_f1[f"{row['feature_set']}_{row['classifier']}"],
    } for row in clf_df.to_dict(orient="records")},
    "per_fold_macro_f1": per_fold,
    "significance": {
        r["classifier"]: {
            "test": r["test"],
            "mean_diff": r["mean_diff"],
            "t_stat": r["t_stat"],
            "p_value": r["p_value"],
            "folds": r["folds"],
        }
        for r in sig_rows
    },
    "wall_clock_seconds": wall_clock,
}
metrics_path = HANDOFF / "metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"wrote {metrics_path}")
print(json.dumps({"best_k": metrics["kmeans"]["best_k"],
                  "best_silhouette": metrics["kmeans"]["best_silhouette"],
                  "best_ari_difficulty": metrics["kmeans"]["ari_vs_difficulty_at_best_k"],
                  "rf_combined_f1": metrics["difficulty_f1"]["combined_rf"]["macro_f1_mean"],
                  "rf_domain_only_f1": metrics["difficulty_f1"]["domain_only_rf"]["macro_f1_mean"],
                  "rf_p_value": metrics["significance"]["rf"]["p_value"]}, indent=2))

wrote /home/ybai/CSCE676/handoff/rq1_text_clustering/metrics.json
{
  "best_k": 4,
  "best_silhouette": 0.09679224342107773,
  "best_ari_difficulty": 0.0438863454937879,
  "rf_combined_f1": 0.5846273189640344,
  "rf_domain_only_f1": 0.33272539363797854,
  "rf_p_value": 1.0623483451783677e-05
}


## Wall-clock summary

Recorded per stage; persisted to `metrics.json["wall_clock_seconds"]`.

In [10]:
print(pd.Series(wall_clock).round(2).to_string())

load_grbench       0.08
embeddings         0.00
kmeans_sweep       2.46
dbscan             0.15
umap              16.42
classification    14.31
